# PLS — implementação na Base 3

O código do walk-forward está em `02_modelos/protocolo_final.py`. Este notebook lê a grade que já foi executada e mostra as decisões que ficaram fixas no teste.

- Alvo: PM2.5 da hora seguinte.
- Preditores: os mesmos 17 do Random Forest, conhecidos na origem.
- Padronização: `scale=True` dentro de cada ajuste.
- Grade: 2, 4, 8, 12, 14 e 16 componentes, escolhida pelo MAE da validação interna. Dezesseis é o maior número testado, porque há 17 preditores.
- Reajuste: uma vez por dia. As horas do dia entram na previsão, mas não no ajuste daquele dia.
- Previsão negativa: truncada em zero.
- O teste não entra na escolha do número de componentes.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

current = Path.cwd()
while current != current.parent and not (current / "bases" / "grupo3").exists():
    current = current.parent
ROOT = current
BASE = ROOT / "analyses" / "grupo3"
OUT = BASE / "03_resultados" / "outputs"
print(ROOT)

tuning = pd.read_csv(BASE / "02_modelos" / "PLS_Regression" / "results" / "tuning_results.csv")
selected = json.loads((BASE / "02_modelos" / "PLS_Regression" / "results" / "selected_config.json").read_text())
meta = json.loads((BASE / "02_modelos" / "PLS_Regression" / "results" / "walkforward_metadata.json").read_text())
print(tuning.to_string(index=False))
print("configuração escolhida", selected["config"])
print("MAE no teste", round(meta["mae_test"], 4), "horas", meta["n_predictions"], "reajuste", meta["refit_every"])


/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation
 config_id           parameters  MAE_validation  n_predictions  seconds
         4 {"n_components": 14}       10.626484           4545      2.3
         5 {"n_components": 16}       10.635005           4545      2.6
         3 {"n_components": 12}       10.657655           4545      2.0
         2  {"n_components": 8}       10.894056           4545      1.4
         1  {"n_components": 4}       13.441022           4545      0.8
         0  {"n_components": 2}       15.602361           4545      0.6
configuração escolhida {'n_components': 14}
MAE no teste 9.6643 horas 6023 reajuste calendar_day


## Um ajuste, para ver o objeto

O bloco abaixo repete um ajuste só no treino interno, com o número de componentes vencedor. Serve para inspecionar pesos e cargas. As previsões do teste continuam sendo as do walk-forward diário, já gravadas em `walkforward_predictions.csv`.

In [2]:
from sklearn.cross_decomposition import PLSRegression

frame = pd.read_csv(BASE / "01_dados" / "outputs" / "features_modelagem_base3.csv")
features = selected["features"]
train = frame[frame["split"] == "train_interno"]
model = PLSRegression(n_components=int(selected["config"]["n_components"]), scale=True, max_iter=500)
model.fit(train[features], train["y"])
print("x_weights_", model.x_weights_.shape)
print("y_loadings_", np.asarray(model.y_loadings_).ravel().round(4))
predictions = pd.read_csv(BASE / "02_modelos" / "PLS_Regression" / "results" / "walkforward_predictions.csv", parse_dates=["forecast_origin"])
print(predictions[["forecast_origin", "y_true", "y_pred", "residual"]].head(3).to_string(index=False))
print("primeira origem", predictions["forecast_origin"].iloc[0], "última", predictions["forecast_origin"].iloc[-1])


x_weights_ (17, 14)
y_loadings_ [ 0.339  -0.1313 -0.094   0.2091 -0.0745  0.1395  0.0546 -0.1416  0.0445
  0.0378 -0.0394 -0.0594 -0.0123  0.0158]
    forecast_origin  y_true   y_pred  residual
2016-05-16 15:00:00    12.0 9.068231  2.931769
2016-05-16 16:00:00    10.0 7.197537  2.802463
2016-05-16 17:00:00    20.0 4.218193 15.781807
primeira origem 2016-05-16 15:00:00 última 2017-02-28 22:00:00


/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/.venv/lib/python3.13/site-packages/scipy/linalg/_basic.py:1462: RuntimeWarning: divide by zero encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/.venv/lib/python3.13/site-packages/scipy/linalg/_basic.py:1462: RuntimeWarning: overflow encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/.venv/lib/python3.13/site-packages/scipy/linalg/_basic.py:1462: RuntimeWarning: invalid value encountered in matmul
  B = (u @ vh[:rank]).conj().T
